In [1]:
import os
import ast
import torch
import torchaudio
import numpy as np
import pandas as pd
import torch.nn as nn
import timm
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from sklearn.isotonic import IsotonicRegression
from tqdm import tqdm

BASE = "../input/competitions/birdclef-2026/"
TRAIN_AUDIO_DIR = os.path.join(BASE, "train_audio")
TRAIN_CSV = os.path.join(BASE, "train.csv")
TAXONOMY_CSV = os.path.join(BASE, "taxonomy.csv")
MODEL_PATH = "../input/models/vladyslavsydorak/dataset-balancing-weights/pytorch/default/1/database_balancing_weights/birdclef_dataset_balancing_epoch0.pth"

class Config:
    SR = 32000               
    DURATION = 5             
    MAX_LENGTH = SR * DURATION 
    N_MELS = 128             
    N_FFT = 1024
    HOP_LENGTH = 512
    BATCH_SIZE = 128
    NUM_WORKERS = 4

taxonomy_df = pd.read_csv(TAXONOMY_CSV)
CLASSES = taxonomy_df['primary_label'].unique().tolist()
NUM_CLASSES = len(CLASSES)
class_to_idx = {c: i for i, c in enumerate(CLASSES)}

In [2]:
class BirdCLEFDataset(Dataset):
    def __init__(self, df, audio_dir, config):
        self.df = df
        self.audio_dir = audio_dir
        self.config = config

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = os.path.join(self.audio_dir, row['filename'])
        waveform, sr = torchaudio.load(audio_path)
        
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
            
        audio_len = waveform.shape[1]
        
        # Center crop for validation
        if audio_len > self.config.MAX_LENGTH:
            start = (audio_len - self.config.MAX_LENGTH) // 2
            waveform = waveform[:, start:start + self.config.MAX_LENGTH]
        elif audio_len < self.config.MAX_LENGTH:
            pad_len = self.config.MAX_LENGTH - audio_len
            waveform = torch.nn.functional.pad(waveform, (0, pad_len))
            
        target = torch.zeros(NUM_CLASSES, dtype=torch.float32)
        primary_idx = class_to_idx.get(row['primary_label'])
        if primary_idx is not None:
            target[primary_idx] = 1.0
            
        secondary_labels = ast.literal_eval(row.get('secondary_labels', "[]"))
        for sec_label in secondary_labels:
            sec_idx = class_to_idx.get(sec_label)
            if sec_idx is not None:
                target[sec_idx] = 1.0

        return waveform, target

In [3]:
from sklearn.model_selection import train_test_split
train_df = pd.read_csv(TRAIN_CSV)
_, val_split = train_test_split(train_df, test_size=0.2, random_state=42)

val_dataset = BirdCLEFDataset(val_split, TRAIN_AUDIO_DIR, Config)
val_loader = DataLoader(
    val_dataset, 
    batch_size=Config.BATCH_SIZE, 
    shuffle=False, 
    num_workers=Config.NUM_WORKERS,
    pin_memory=True,
    prefetch_factor=2
)

# 4. Initialize Model and Transforms
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
class BirdCLEFModel(nn.Module):
    def __init__(self, model_name='efficientnet_b0', num_classes=234):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=False, num_classes=0)
        self.head = nn.Linear(self.backbone.num_features, num_classes)
    def forward(self, x):
        return self.head(self.backbone(x))

In [5]:
model = BirdCLEFModel(num_classes=NUM_CLASSES)

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

model.load_state_dict(torch.load(MODEL_PATH))
model.to(device)
model.eval()

mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=Config.SR, n_fft=Config.N_FFT, hop_length=Config.HOP_LENGTH, 
    n_mels=Config.N_MELS, f_min=50, f_max=14000
).to(device)
amp_to_db = torchaudio.transforms.AmplitudeToDB().to(device)

In [6]:
print("Generating OOF predictions for threshold calibration...")
all_val_targets = []
all_val_preds = []

with torch.no_grad():
    for waveforms, targets in tqdm(val_loader):
        waveforms = waveforms.to(device)
        targets = targets.to(device)
        
        mel_spec = mel_transform(waveforms)
        mel_spec = amp_to_db(mel_spec)
        
        batch_size = mel_spec.size(0)
        mel_flat = mel_spec.reshape(batch_size, -1)
        mins = mel_flat.min(dim=1, keepdim=True)[0].reshape(batch_size, 1, 1, 1)
        maxs = mel_flat.max(dim=1, keepdim=True)[0].reshape(batch_size, 1, 1, 1)
        
        mel_spec = (mel_spec - mins) / (maxs - mins + 1e-6)
        mel_spec = mel_spec * 2 - 1
        images = mel_spec.expand(-1, 3, -1, -1)
        
        probs = torch.sigmoid(model(images))
        
        all_val_targets.append(targets.cpu().numpy())
        all_val_preds.append(probs.cpu().numpy())

y_true = np.vstack(all_val_targets)
y_prob = np.vstack(all_val_preds)


Generating OOF predictions for threshold calibration...


100%|██████████| 56/56 [01:36<00:00,  1.73s/it]


In [7]:
def optimize_thresholds(y_true, y_prob):
    thresholds = np.full(NUM_CLASSES, 0.5, dtype=np.float32)
    threshold_grid = [0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]
    
    for c in tqdm(range(NUM_CLASSES), desc="Calibrating Thresholds"):
        true_c = y_true[:, c]
        prob_c = y_prob[:, c]
        
        if true_c.sum() < 3:
            continue
            
        try:
            ir = IsotonicRegression(out_of_bounds="clip")
            ir.fit(prob_c, true_c)
            calibrated_probs = ir.transform(prob_c)
        except Exception:
            calibrated_probs = prob_c
            
        best_f1, best_t = 0.0, 0.5
        for t in threshold_grid:
            preds = (calibrated_probs >= t).astype(int)
            f1 = f1_score(true_c, preds, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_t = t
                
        thresholds[c] = best_t
        
    return thresholds

In [8]:
optimal_thresholds = optimize_thresholds(y_true, y_prob)
np.save('optimal_thresholds.npy', optimal_thresholds)
print(f"Mean optimized threshold: {optimal_thresholds.mean():.3f}")

Calibrating Thresholds: 100%|██████████| 234/234 [00:04<00:00, 52.46it/s]

Mean optimized threshold: 0.328
